# 1. Data cleaning

Cleans the MSBase and SMSReg visit data and splits them into train / calibration / test sets (70/10/20 by patient).
Both registries go through the same steps but are never mixed.

Input files in `RAW_DIR` (see `dummy_data/` for an example):
- `msbase_visits.csv`, `smsreg_visits.csv`: one row per visit with `patient_code, country, visit_date, birth_date, sex (F/M), diagnosis_date, disease_course (RR/SP), edss_score, edss_pyramidal, treatment_class`. `treatment_class` is one of `first_line, second_line, none, hsct, other`. SMSReg doesn't need `country`.
- `msbase_relapses.csv`, `smsreg_relapses.csv`: `patient_code, relapse_date` (only used for the Lorscheider criteria).

Everything is written to `custom_run/`.

In [1]:
import os

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from harmonise_utils import FEATURES

In [2]:
RAW_DIR = "dummy_data"
RUN_DIR = "custom_run"

SEED = 42
TEST_SIZE = 0.2
CALIB_SIZE = 0.1

# female = 1, as in the original code
SEX_MAP = {"F": 1, "M": 0}

# a country is used for training if it has > 1000 visits, >= 50 pwRRMS and >= 50 pwSPMS + pwMS-T
MIN_VISITS = 1000
MIN_RRMS = 50
MIN_SPMS = 50

KEEP_COLUMNS = ["patient_code", "country", "visit_date", "edss_pyramidal", "y_label"] + FEATURES

## Cleaning

- keep visits with EDSS, age, sex and an RRMS/SPMS diagnosis
- drop RRMS visits with EDSS >= 9 and SPMS visits with EDSS <= 2
- keep complete cases for the nine features
- for patients who are RRMS throughout, drop visits in the last 3 years of follow-up (they may have transitioned without it being recorded yet)

In [3]:
def show(step, df):
    print(f"  {step:<45} {df['patient_code'].nunique():>7,} patients {len(df):>9,} visits")


def add_features(raw):
    df = raw.copy()
    for col in ["visit_date", "birth_date", "diagnosis_date"]:
        df[col] = pd.to_datetime(df[col])

    df["age_at_visit"] = (df["visit_date"] - df["birth_date"]).dt.days / 365.25
    df["age_at_diagnosis"] = (df["diagnosis_date"] - df["birth_date"]).dt.days / 365.25
    df["year_diagnosis"] = df["diagnosis_date"].dt.year
    df["sex_label"] = df["sex"].map(SEX_MAP)

    treatment = df["treatment_class"].replace({"other": "none"})  # other drugs count as no treatment
    df["first_line_DMT"] = (treatment == "first_line").astype(int)
    df["second_line_DMT"] = (treatment == "second_line").astype(int)
    df["no_treatment"] = (treatment == "none").astype(int)
    df["stem_cell_treatment"] = (treatment == "hsct").astype(int)
    df.loc[df["treatment_class"].isna(), ["first_line_DMT", "second_line_DMT", "no_treatment", "stem_cell_treatment"]] = np.nan

    df["y_label"] = df["disease_course"].map({"RR": 0, "SP": 1})
    return df


def patient_groups(df):
    labels = df.groupby("patient_code")["y_label"].agg(lambda s: frozenset(s.astype(int)))
    return labels.map({frozenset({0}): "RRMS", frozenset({1}): "SPMS", frozenset({0, 1}): "transition"})


def clean(df):
    show("raw", df)
    df = df.dropna(subset=["edss_score", "age_at_visit", "sex_label", "y_label"])
    show("EDSS, age, sex, RRMS/SPMS present", df)
    df = df[((df["y_label"] == 0) & (df["edss_score"] <= 8.5)) | ((df["y_label"] == 1) & (df["edss_score"] >= 2.5))]
    show("EDSS exclusions", df)
    df = df.dropna(subset=FEATURES)
    show("complete features", df)

    groups = patient_groups(df)
    last_visit = df.groupby("patient_code")["visit_date"].transform("max")
    recent = df["patient_code"].isin(groups[groups == "RRMS"].index) & (df["visit_date"] > last_visit - pd.DateOffset(years=3))
    df = df[~recent]
    show("drop last 3 years of pwRRMS", df)

    df = df.sort_values(["patient_code", "visit_date"]).reset_index(drop=True)
    int_cols = ["y_label", "sex_label", "year_diagnosis", "first_line_DMT", "second_line_DMT", "no_treatment", "stem_cell_treatment"]
    df[int_cols] = df[int_cols].astype(int)
    df["patient_group"] = df["patient_code"].map(patient_groups(df))
    return df

## Data sufficiency and splits

Countries that don't meet the criteria go entirely to the test set. SMSReg is always used for training.
The split is stratified by patient group (pwRRMS / pwSPMS / pwMS-T).

In [4]:
def sufficiency_table(df, registry):
    rows = []
    for country, sub in df.groupby("country"):
        groups = sub.drop_duplicates("patient_code")["patient_group"]
        rows.append({"registry": registry, "country": country, "visits": len(sub),
                     "pw_rrms": (groups == "RRMS").sum(), "pw_spms": (groups == "SPMS").sum(),
                     "pw_ms_t": (groups == "transition").sum()})
    table = pd.DataFrame(rows)
    enough = ((table["visits"] > MIN_VISITS) & (table["pw_rrms"] >= MIN_RRMS)
              & (table["pw_spms"] + table["pw_ms_t"] >= MIN_SPMS))
    table["trainable"] = enough | (registry == "smsreg")
    return table


def split_patients(groups):
    patients = groups.index.to_numpy()
    calib_share = CALIB_SIZE / (1 - TEST_SIZE)
    try:
        rest, test = train_test_split(patients, test_size=TEST_SIZE, stratify=groups.to_numpy(), random_state=SEED)
        train, calib = train_test_split(rest, test_size=calib_share, stratify=groups.loc[rest].to_numpy(), random_state=SEED)
    except ValueError:  # some group too small to stratify
        rest, test = train_test_split(patients, test_size=TEST_SIZE, random_state=SEED)
        train, calib = train_test_split(rest, test_size=calib_share, random_state=SEED)
    return train, calib, test


def assign_splits(df, trainable):
    split = {}
    for country, sub in df.groupby("country"):
        groups = sub.drop_duplicates("patient_code").set_index("patient_code")["patient_group"]
        if country in trainable:
            for name, patients in zip(["train", "calib", "test"], split_patients(groups)):
                split.update(dict.fromkeys(patients, name))
        else:
            split.update(dict.fromkeys(groups.index, "test"))
    return df["patient_code"].map(split)

## Run

In [5]:
sufficiency = []

for registry in ["msbase", "smsreg"]:
    print(registry)
    raw = pd.read_csv(f"{RAW_DIR}/{registry}_visits.csv")
    if registry == "smsreg":
        raw["country"] = "SE"
    if "edss_pyramidal" not in raw.columns:
        raw["edss_pyramidal"] = np.nan
    keep = KEEP_COLUMNS.copy()
    if "DUMMY_DATA" in raw.columns:
        print("  (dummy data)")
        keep.append("DUMMY_DATA")

    df = clean(add_features(raw))
    suff = sufficiency_table(df, registry)
    trainable = sorted(suff.loc[suff["trainable"], "country"])
    print("  trainable:", trainable)
    df["split"] = assign_splits(df, trainable)

    out = f"{RUN_DIR}/data/{registry}"
    os.makedirs(out, exist_ok=True)
    for name in ["train", "calib", "test"]:
        part = df[df["split"] == name]
        part[keep].to_csv(f"{out}/{name}.csv", index=False)
        show(name, part)

    relapses = pd.read_csv(f"{RAW_DIR}/{registry}_relapses.csv")
    relapses = relapses[relapses["patient_code"].isin(df["patient_code"])]
    relapses[["patient_code", "relapse_date"]].to_csv(f"{out}/relapses.csv", index=False)

    sufficiency.append(suff)

sufficiency = pd.concat(sufficiency, ignore_index=True)
sufficiency.to_csv(f"{RUN_DIR}/data/data_sufficiency.csv", index=False)
sufficiency

msbase
  (dummy data)
  raw                                             1,020 patients     8,253 visits
  EDSS, age, sex, RRMS/SPMS present               1,020 patients     8,253 visits
  EDSS exclusions                                 1,020 patients     8,253 visits
  complete features                               1,020 patients     8,253 visits
  drop last 3 years of pwRRMS                       955 patients     5,900 visits
  trainable: ['AU', 'CA', 'IT']
  train                                             588 patients     3,681 visits
  calib                                              85 patients       552 visits
  test                                              282 patients     1,667 visits
smsreg
  (dummy data)
  raw                                               400 patients     3,179 visits
  EDSS, age, sex, RRMS/SPMS present                 400 patients     3,179 visits
  EDSS exclusions                                   400 patients     3,179 visits
  complete features   

,registry,country,visits,pw_rrms,pw_spms,pw_ms_t,trainable
0,msbase,AR,315,33,14,10,False
1,msbase,AU,1733,157,62,61,True
2,msbase,CA,1785,157,60,62,True
3,msbase,IT,1720,155,74,54,True
4,msbase,JP,347,28,16,12,False
5,smsreg,SE,2215,219,80,70,True
